# Module 6 project: QAOA trained on a simulator, evaluated with noise

**Quantum Computing Intermediate · Module 6 · about 180 minutes** · notebook version 2026-10-05

**The question of this experiment.** In Module 5, a second QAOA layer gave much better cuts. Each layer adds one `rzz` gate per edge, and on a noisy computer every gate costs a little. A common way to use QAOA is to find good angles on a simulator without noise, then run the circuit on the computer. **When angles trained without noise are run with noise, how much of each layer's gain survives, and how noisy can the computer be before a third layer stops helping?**

In this project you run an experiment on a **noisy simulator**: a simulator that adds the errors of a real quantum computer, described by a **noise model**. You compare what you measure with what the noise model predicts, say how certain each result is, and say what the experiment cannot tell you. These are the skills of course outcome 5.

**The plan.**

1. Write three statistics functions (Step 1) and meet the course noise model (Step 2).
2. Write `qaoa_circuit(gammas, betas, edges, n)` (your Module 5 code) and `mean_cut(counts, edges)` for a 5-node graph (Step 3).
3. Train 1, 2 and 3 layers on the ideal simulator (Step 4).
4. Run the trained circuit with noise and compare with the noise model's expected values (Step 5).
5. Compare 1, 2 and 3 layers with noise, with uncertainties (Step 6), and find how much noise a third layer can take (Step 7).
6. Run your own computer's noise model and get your verification value (Step 8), then look at the limits.

**How it is graded.** Your project quiz in Canvas (the one for this experiment, 20 points, pass at 16) asks for your verification value from the check cell, three results from this notebook, and four questions about uncertainty, interpretation and limits. Your project quiz gives you three numbers: LAYERS (1, 2 or 3), and P2 and READOUT for your computer.

**Saving your work.** JupyterLite keeps your changes in this browser, so you can stop and come back. To keep a copy elsewhere, use **File → Download**. Run each code cell with **Shift + Enter**, in order; if you come back later, run the cells from the top again.

## Step 0: start Python

Run the cell below. The first time, Python can take up to a minute to start in your browser, and Matplotlib takes a few more seconds to load.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import qsim
from qsim import (QuantumCircuit, Statevector, AerSimulator, NoiseModel, depolarizing_error, ReadoutError,
                  simulate_density_matrix)
from scipy.optimize import minimize

print("Ready. qsim", qsim.__version__, "and NumPy", np.__version__)

## Step 1: how certain is a result from shots?

A circuit that gives a result with probability p, run for S shots, gives that result k times. The number k is random: it follows the **binomial distribution**. The fraction f = k / S has

- mean p, and
- standard deviation **σ = √(p (1 − p) / S)**. This spread is called **shot noise**.

For example, with S = 4,000 shots and p = 0.587, σ = 0.0078: two runs of the same circuit typically differ by about this much. Four times as many shots halve σ.

**The course rule: 3σ.** A measured fraction f agrees with an expected value p when |f − p| ≤ 3σ. For a noisy simulator, p must be the **noise model's** expected value (computed exactly, without shots, in Step 2), not the ideal value from a circuit without noise. A correct run lands outside 3σ only about 3 times in 1,000 (0.27%).

**Comparing two results.** Two fractions f₁ and f₂, each from S independent shots, differ by f₂ − f₁, with standard deviation

**σ_diff = √( f₁(1 − f₁)/S + f₂(1 − f₂)/S )**.

Uncertainties of independent results add **in quadrature** (their squares add), so a difference is less certain than either result alone. A difference larger than 3σ_diff is **significant**: shot noise alone would almost never produce it. A smaller difference may be real, but this experiment cannot show it.

**Your task:** write the three functions below. They are used in every later step, and the check cell tests them.

- `shot_sigma(p, shots)`: σ = √(p (1 − p) / shots), the standard deviation of a fraction.
- `within_3_sigma(k, shots, p)`: `True` if the fraction k / shots is within 3 · shot_sigma(p, shots) of p (in either direction), otherwise `False`.
- `difference_sigma(f1, f2, shots)`: σ_diff for two fractions from `shots` shots each.

In [ ]:
def shot_sigma(p, shots):
    """The standard deviation of a fraction measured with `shots` shots, when the true probability is p."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete shot_sigma() first.")


def within_3_sigma(k, shots, p):
    """True if k successes out of `shots` agree with the probability p by the course's 3-sigma rule."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete within_3_sigma() first.")


def difference_sigma(f1, f2, shots):
    """The standard deviation of f2 - f1, for two fractions from `shots` independent shots each."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete difference_sigma() first.")

In [ ]:
print("shot_sigma(0.687, 4000) =", round(shot_sigma(0.687, 4000), 5), "  (expected 0.00733)")
print("within_3_sigma(2748, 4000, 0.687) =", within_3_sigma(2748, 4000, 0.687), "  (expected True: exactly 0.687)")
print("within_3_sigma(2655, 4000, 0.687) =", within_3_sigma(2655, 4000, 0.687), "  (expected False: 3.2 sigma too low)")
print("difference_sigma(0.687, 0.634, 4000) =", round(difference_sigma(0.687, 0.634, 4000), 5), "  (expected 0.01057)")

## Step 2: the course noise model

A **noise model** says what can go wrong in each operation. The course model has three parts, each with a number:

- after every **two-qubit gate** (`cx`, `cz`, `cp`, `rzz`, `swap`), a **depolarizing error** with parameter P2: with probability P2 the two qubits are replaced by a completely random state. For a density matrix, ρ → (1 − P2) ρ + P2 · I/4 on those two qubits. (An average gate error, the number IBM reports, of 0.75 · P2.)
- after every **one-qubit gate**, the same on one qubit with parameter P2 / 10: one-qubit gates are about ten times better.
- at **readout**: a qubit in |0⟩ is recorded as 1 with probability READOUT, and a qubit in |1⟩ is recorded as 0 with probability 2 · READOUT. The second is larger because an excited qubit can decay to |0⟩ during the measurement.

The course values are **P2 = 0.01** and **READOUT = 0.02**. They are several times worse than IBM's best current computers (the model of `ibm_pittsburgh` used in Module 4 has a median two-qubit gate error of 0.0015 and a median readout error of 0.0045), so that the effects of noise show clearly in a few thousand shots. In Canvas, your project quiz gives you your own P2 (0.005 to 0.030) and READOUT (0.005 to 0.030): a model of "your" computer.

The noise model applies errors by **gate name**. A gate on three qubits, such as `ccx` or `mcx`, would get no error at all, so every circuit in this project uses only one- and two-qubit gates.

The prepared cell builds the noise model with the same functions as Qiskit Aer (`qiskit_aer.noise`), and defines `expected_probabilities(qc, p2, readout)`. It gives the noise model's **exact** probability of every recorded result, with no shots and no randomness: it follows the **density matrix**, the mixed state that the noisy gates produce, then applies the readout errors to the recorded bits. A run with shots, `AerSimulator(noise_model=...).run(qc, shots=4000)`, samples from these same probabilities and adds shot noise. The 3σ rule compares the two.

In [ ]:
ONE_QUBIT_GATES = ["h", "x", "rx", "ry", "rz", "p", "t", "tdg", "s", "sdg", "sx"]
TWO_QUBIT_GATES = ["cx", "cz", "cp", "rzz", "swap"]


def course_noise_model(p2=0.01, readout=0.02):
    """Prepared: depolarizing error p2 after two-qubit gates and p2/10 after one-qubit gates; readout errors."""
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p2 / 10, 1), ONE_QUBIT_GATES)
    nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), TWO_QUBIT_GATES)
    nm.add_all_qubit_readout_error(ReadoutError([[1 - readout, readout], [2 * readout, 1 - 2 * readout]]))
    return nm


def expected_probabilities(qc, p2=0.01, readout=0.02):
    """Prepared: the noise model's exact probability of each recorded result, as a dict {"bits": probability}.

    qc must end with its measurements (classical bit j receives one qubit). No shots are taken.
    """
    measured = {inst.clbits[0]: inst.qubits[0] for inst in qc.data if inst.name == "measure"}
    body = qc.copy()
    body.remove_final_measurements()
    rho = simulate_density_matrix(body, course_noise_model(p2, readout))   # the exact mixed state after the noisy gates
    m = len(measured)
    probs = rho.probabilities(qargs=[measured[j] for j in range(m)])      # index bit j = classical bit j
    confusion = np.array([[1 - readout, readout], [2 * readout, 1 - 2 * readout]])   # rows: true bit; columns: recorded bit
    for j in range(m):                                                     # each recorded bit can be flipped
        t = probs.reshape(2 ** (m - 1 - j), 2, 2 ** j)
        probs = np.einsum("aib,ij->ajb", t, confusion).reshape(-1)
    return {format(i, f"0{m}b"): float(p) for i, p in enumerate(probs) if p > 1e-15}


print(course_noise_model())

## Step 3: the project graph, the circuit and the measured cut

The project graph has 5 nodes: a ring 0-1-2-3-4-0 and one chord, (0, 2). It has 6 edges. The cell finds the maximum cut by brute force over all 2⁵ = 32 splits: bit q of a split (counted from the right) is node q's group.

**Your task (a):** write `qaoa_circuit(gammas, betas, edges, n)` as in Module 5: H on every qubit; then, for each layer k, `rzz(2 * gammas[k], i, j)` on every edge and `rx(2 * betas[k], q)` on every qubit. No measurements: the training uses the state itself, and the prepared `measured(qc)` adds the measurements for the runs with shots.

**Your task (b):** write `mean_cut(counts, edges)`: the average cut size over all shots. For each result key, node q's group is `key[-1 - q]` (qubit 0 is the rightmost character); an edge (i, j) is cut when the two groups differ. Weight each key by its count and divide by the total number of shots.

In [ ]:
EDGES = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0), (0, 2)]
N = 5
CUTS = np.array([sum(((s >> i) & 1) != ((s >> j) & 1) for i, j in EDGES) for s in range(2 ** N)])   # cut of each split
MAX_CUT = CUTS.max()
print("maximum cut:", MAX_CUT, "for", [format(s, "05b") for s in range(2 ** N) if CUTS[s] == MAX_CUT])
print("average cut of a random split:", CUTS.mean())


def qaoa_circuit(gammas, betas, edges, n):
    """QAOA: H on every qubit, then for each layer rzz(2*gamma) on every edge and rx(2*beta) on every qubit."""
    qc = QuantumCircuit(n)
    # YOUR CODE HERE
    raise NotImplementedError("Complete qaoa_circuit() first.")
    return qc


def mean_cut(counts, edges):
    """The average cut size over all shots in counts (node q is the bit key[-1 - q])."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete mean_cut() first.")


def measured(qc):
    """Prepared: a copy of qc that measures every qubit (qubit q into classical bit q)."""
    out = QuantumCircuit(qc.num_qubits, qc.num_qubits)
    out.compose(qc, inplace=True)
    out.measure(list(range(qc.num_qubits)), list(range(qc.num_qubits)))
    return out

In [ ]:
print("mean_cut({'01001': 3, '00000': 1}, EDGES) =", mean_cut({"01001": 3, "00000": 1}, EDGES), "  (expected 3.75)")
qc = qaoa_circuit([0.4], [0.3], EDGES, N)
print("gates in one layer:", qc.count_ops())

**What to notice.** The maximum cut is 5 of the 6 edges: the chord makes a triangle (0, 1, 2), and a triangle can never have all three edges cut. Four splits reach it. A random split cuts 3 edges on average, so any useful QAOA result must beat 3. One layer has 6 `rzz` gates (one per edge) and 5 `rx` gates.

## Step 4: train on the ideal simulator

The cell trains the angles without noise, as in Module 5, with the **expected cut** computed from the state vector:

- **1 layer:** a grid scan of 33 × 17 angles (γ from 0 to π, β from 0 to π/2), then COBYLA from the best grid point;
- **2 layers:** COBYLA started from the 1-layer angles, repeated (γ₁, γ₁, β₁, β₁): a **warm start**;
- **3 layers:** COBYLA started from the 2-layer angles, with the second layer repeated.

It takes about half a minute. COBYLA's path depends on the SciPy version, so your angles can differ a little from the course's **reference angles**, printed below; the expected cuts agree to about 3 decimals. From Step 5 on, everyone uses the reference angles, so that the numbers in the project quiz are the same for all.

In [ ]:
def ideal_cut(x):
    """The expected cut without noise, for x = [gamma_1, ..., gamma_p, beta_1, ..., beta_p]."""
    p = len(x) // 2
    probs = Statevector(qaoa_circuit(list(x[:p]), list(x[p:]), EDGES, N)).probabilities()
    return float(probs @ CUTS)


gammas_grid, betas_grid = np.linspace(0, np.pi, 33), np.linspace(0, np.pi / 2, 17)
grid = np.array([[ideal_cut([g, b]) for g in gammas_grid] for b in betas_grid])
k = np.unravel_index(np.argmax(np.round(grid, 9)), grid.shape)
r1 = minimize(lambda x: -ideal_cut(x), [gammas_grid[k[1]], betas_grid[k[0]]], method="COBYLA", options={"maxiter": 300})
r2 = minimize(lambda x: -ideal_cut(x), [r1.x[0], r1.x[0], r1.x[1], r1.x[1]], method="COBYLA", options={"maxiter": 800})
r3 = minimize(lambda x: -ideal_cut(x), [r2.x[0], r2.x[1], r2.x[1], r2.x[2], r2.x[3], r2.x[3]], method="COBYLA",
              options={"maxiter": 1500})

REFERENCE_ANGLES = {1: ([2.817], [0.354]),
                    2: ([2.849, 2.627], [0.517, 0.309]),
                    3: ([2.905, 2.669, 2.619], [0.535, 0.401, 0.204])}
for p, r in ((1, r1), (2, r2), (3, r3)):
    g, b = REFERENCE_ANGLES[p]
    print(f"{p} layer(s): your training {-r.fun:.4f} at angles {np.round(r.x, 3)} | reference angles {ideal_cut(g + b):.4f}")

**What to notice.** Without noise each layer helps: the expected cut rises from 4.110 (1 layer) to 4.623 (2) and 4.826 (3), out of a maximum of 5. The warm start matters: COBYLA started for 2 layers from all angles 0.5 stops at about 4.16, on a lower hill of the landscape (Module 5).

## Step 5: the trained circuit with noise

Now run the 2-layer circuit with the reference angles on the course noise model. Two numbers describe each run:

- the **mean cut** over the shots, compared with the noise model's expected cut; its uncertainty is the standard error of the mean, SEM = s / √shots, where s is the standard deviation of the cut over the shots (prepared: `cut_sem`);
- the fraction of shots that gave a **maximum cut**, a binomial fraction with σ from your `shot_sigma`, tested with `within_3_sigma`.

The project quiz asks for your measured mean cut (any run with 4,000 shots) and for the expected fraction of maximum cuts.

In [ ]:
def cut_sem(counts, edges):
    """Prepared: the standard error of the mean cut: (standard deviation of the per-shot cut) / sqrt(shots)."""
    cuts = np.array([sum(k[-1 - i] != k[-1 - j] for i, j in edges) for k in counts], dtype=float)
    n = np.array([counts[k] for k in counts], dtype=float)
    mean = np.sum(n * cuts) / n.sum()
    return float(np.sqrt(np.sum(n * (cuts - mean) ** 2) / (n.sum() - 1) / n.sum()))


def expected_values(layers, p2, readout):
    """Prepared: the noise model's expected cut and probability of a maximum cut for the reference angles."""
    g, b = REFERENCE_ANGLES[layers]
    probs = expected_probabilities(measured(qaoa_circuit(g, b, EDGES, N)), p2, readout)
    cut = sum(p * CUTS[int(s, 2)] for s, p in probs.items())
    pmax = sum(p for s, p in probs.items() if CUTS[int(s, 2)] == MAX_CUT)
    return cut, pmax


RUN_SEED = 1                     # any whole number; change it to see another run
sim = AerSimulator(noise_model=course_noise_model(0.01, 0.02))
g, b = REFERENCE_ANGLES[2]
qc = measured(qaoa_circuit(g, b, EDGES, N))
cut_exp, pmax_exp = expected_values(2, 0.01, 0.02)
counts = sim.run(qc, shots=4000, seed_simulator=RUN_SEED).result().get_counts()
kmax = sum(v for s, v in counts.items() if CUTS[int(s, 2)] == MAX_CUT)
print(f"mean cut: ideal {ideal_cut(g + b):.4f}, noise model expected {cut_exp:.4f}, measured {mean_cut(counts, EDGES):.4f} ± {cut_sem(counts, EDGES):.4f} (SEM)")
print(f"maximum cut: ideal 0.7070, expected {pmax_exp:.4f}, measured {kmax / 4000:.4f}, sigma {shot_sigma(pmax_exp, 4000):.4f}, "
      f"within 3 sigma: {within_3_sigma(kmax, 4000, pmax_exp)}")

**What to notice.** With noise the 2-layer circuit gives an expected cut of 4.333 instead of 4.623: it keeps about 82% of its gain over a random split (1.333 of 1.623). The maximum cut now comes out 59% of the time instead of 71%. The noise pushes the distribution toward random splits, whose average cut is 3.

## Step 6: 1, 2 or 3 layers with noise?

The cell runs the three trained circuits on the course noise model, 4,000 shots each, prints the expected and measured mean cuts, and tests the measured differences between neighbouring layer counts with σ_diff = √(SEM₁² + SEM₂²).

In [ ]:
rows = {}
for p in (1, 2, 3):
    g, b = REFERENCE_ANGLES[p]
    cts = sim.run(measured(qaoa_circuit(g, b, EDGES, N)), shots=4000, seed_simulator=RUN_SEED + 10 + p).result().get_counts()
    cut_exp, pmax_exp = expected_values(p, 0.01, 0.02)
    rows[p] = (ideal_cut(g + b), cut_exp, mean_cut(cts, EDGES), cut_sem(cts, EDGES))
print("layers  rzz gates   ideal    expected (noise)   measured ± SEM")
for p, (ic, ec, mc, se) in rows.items():
    print(f"  {p}        {6 * p:2d}      {ic:.4f}       {ec:.4f}         {mc:.4f} ± {se:.4f}")
for a, b2 in ((1, 2), (2, 3)):
    d, sd = rows[b2][2] - rows[a][2], math.sqrt(rows[a][3] ** 2 + rows[b2][3] ** 2)
    print(f"{a} -> {b2} layers: measured gain {d:+.4f}, sigma_diff {sd:.4f} = {d / sd:+.1f} sigma_diff,",
          "significant" if abs(d) > 3 * sd else "not significant")

plt.figure(figsize=(6.5, 3.4))
plt.plot([1, 2, 3], [rows[p][0] for p in (1, 2, 3)], "k--o", label="ideal")
plt.plot([1, 2, 3], [rows[p][1] for p in (1, 2, 3)], "C0-s", label="noise model, expected")
plt.errorbar([1, 2, 3], [rows[p][2] for p in (1, 2, 3)], yerr=[3 * rows[p][3] for p in (1, 2, 3)], fmt="C3o", capsize=4, label="4000 shots, ±3 SEM")
plt.axhline(3, color="gray", ls=":", lw=1, label="random split")
plt.xticks([1, 2, 3]); plt.xlabel("layers"); plt.ylabel("mean cut"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

**What to notice.** Under the course noise model the third layer still helps: the expected gain from 2 to 3 layers is 0.111 (ideal: 0.203), about 5σ_diff with 4,000 shots each, so your measured gain will almost always be significant. But noise takes a larger share of each new layer's gain: of the gain that each layer adds without noise, it keeps about 85% for the first layer (over a random split), 75% for the second and 55% for the third.

## Step 7: how much noise can a third layer take?

The cell scans P2 from 0 to 0.050 in steps of 0.001 (READOUT stays 0.02) and computes the expected cut for 2 and 3 layers; no shots. It prints the **crossover**: the smallest P2 in the scan at which 2 layers give a higher expected cut than 3. The project quiz asks for it. It takes about half a minute.

In [ ]:
p2_values = np.round(np.arange(0, 0.0505, 0.001), 3)
two = [expected_values(2, p, 0.02)[0] for p in p2_values]
three = [expected_values(3, p, 0.02)[0] for p in p2_values]
crossover = next(p for p, a, b in zip(p2_values, two, three) if a > b)
print(f"crossover: from P2 = {crossover:.3f}, 2 layers give a higher expected cut than 3")
i = list(p2_values).index(0.03)
print(f"at P2 = 0.030: 2 layers {two[i]:.4f}, 3 layers {three[i]:.4f}, difference {three[i] - two[i]:+.4f}")

plt.figure(figsize=(7, 3.3))
plt.plot(p2_values, two, label="2 layers"); plt.plot(p2_values, three, label="3 layers")
plt.axvline(crossover, color="gray", ls=":", lw=1)
plt.xlabel("P2 (two-qubit depolarizing error)"); plt.ylabel("expected cut"); plt.legend(); plt.tight_layout(); plt.show()

**What to notice.** The curves cross near P2 = 0.032. At P2 = 0.030, the highest value in the personal range, 3 layers are still better by 0.006 in expectation, but that is far below σ_diff (about 0.024 for 4,000 shots each): no affordable experiment would show it. In practice, on a computer near the crossover the extra layer costs gates, angles to train and quantum time, and gives nothing measurable.

## Step 8: your computer, and your verification value

Open your **project quiz for this experiment** in Canvas. Question 1 shows your LAYERS (1, 2 or 3), your P2 and your READOUT. Type them below and run the cells.

The first cell runs the circuit with the reference angles for LAYERS on **your** noise model with 4,000 shots and keeps the counts as `my_counts`. The check cell tests your three statistics functions, `qaoa_circuit()` and `mean_cut()`, and checks that `my_counts` agree with your noise model by the 3σ rule (for the mean cut, 3 SEM). Only if everything passes does it print your **verification value**: 1,000 times the noise model's exact expected cut on your computer, rounded to a whole number.

In [ ]:
LAYERS = 0       # your number from Canvas: 1, 2 or 3
P2 = 0.0         # your number from Canvas, for example 0.017
READOUT = 0.0    # your number from Canvas, for example 0.015
my_counts = {}   # filled by the next cell

In [ ]:
g, b = REFERENCE_ANGLES[int(LAYERS)]
my_counts = AerSimulator(noise_model=course_noise_model(P2, READOUT)).run(
    measured(qaoa_circuit(g, b, EDGES, N)), shots=4000, seed_simulator=RUN_SEED + 100).result().get_counts()
my_cut, my_pmax = expected_values(int(LAYERS), P2, READOUT)
print(f"{int(LAYERS)} layer(s) on your computer: expected cut {my_cut:.4f}, measured {mean_cut(my_counts, EDGES):.4f} ± {cut_sem(my_counts, EDGES):.4f} (SEM)")
print(f"expected fraction of maximum cuts: {my_pmax:.4f}")

In [ ]:
# The course's check code. You do not need to read it; it is here so that you can.
import math

import numpy as np

try:                                    # real Qiskit, if it is installed
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
except ImportError:                     # the browser: the course simulator
    from qsim import QuantumCircuit, Statevector

ONE_QUBIT_GATES = ["h", "x", "rx", "ry", "rz", "p", "t", "tdg", "s", "sdg", "sx"]
TWO_QUBIT_GATES = ["cx", "cz", "cp", "rzz", "swap"]
SHOTS = 4000

# ---------------------------------------------------------------- an exact noisy simulator (NumPy only)
_S2 = 1 / math.sqrt(2)
_FIXED = {
    "h": np.array([[_S2, _S2], [_S2, -_S2]], dtype=complex),
    "x": np.array([[0, 1], [1, 0]], dtype=complex),
    "t": np.diag([1, np.exp(1j * math.pi / 4)]),
    "tdg": np.diag([1, np.exp(-1j * math.pi / 4)]),
    "s": np.diag([1, 1j]),
    "sdg": np.diag([1, -1j]),
}


def _matrix(name, params):
    """The gate's matrix in Qiskit's convention (for two-qubit gates, the first qubit is the right-hand factor)."""
    if name in _FIXED:
        return _FIXED[name]
    if name == "rx":
        c, s = math.cos(params[0] / 2), math.sin(params[0] / 2)
        return np.array([[c, -1j * s], [-1j * s, c]])
    if name == "rz":
        return np.diag([np.exp(-0.5j * params[0]), np.exp(0.5j * params[0])])
    if name == "p":
        return np.diag([1, np.exp(1j * params[0])])
    if name == "cx":                     # control = first qubit (the low bit of the 4 x 4 index)
        return np.array([[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0]], dtype=complex)
    if name == "cz":
        return np.diag([1, 1, 1, -1]).astype(complex)
    if name == "cp":
        return np.diag([1, 1, 1, np.exp(1j * params[0])])
    if name == "rzz":
        a = np.exp(-0.5j * params[0])
        b = np.exp(0.5j * params[0])
        return np.diag([a, b, b, a])
    if name == "swap":
        return np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1]], dtype=complex)
    raise ValueError(f"unknown gate {name}")


def _apply(rho, u, qubits, n):
    """rho -> U rho U^dagger, with U acting on the given qubits (qubits[0] is the low bit of U's index)."""
    k = len(qubits)
    t = rho.reshape([2] * (2 * n))
    # axis of qubit q in the row index is n - 1 - q; in the column index 2n - 1 - q
    ut = u.reshape([2] * (2 * k))           # axes: out bits (high to low), in bits (high to low)
    row_axes = [n - 1 - q for q in reversed(qubits)]
    col_axes = [2 * n - 1 - q for q in reversed(qubits)]
    t = np.tensordot(ut, t, axes=(list(range(k, 2 * k)), row_axes))
    t = np.moveaxis(t, list(range(k)), row_axes)
    t = np.tensordot(t, ut.conj(), axes=(col_axes, list(range(k, 2 * k))))
    t = np.moveaxis(t, list(range(2 * n - k, 2 * n)), col_axes)
    return t.reshape(2 ** n, 2 ** n)


def _depolarize(rho, lam, qubits, n):
    """rho -> (1 - lam) rho + lam (I / 2^k on the given qubits) x (rho traced over them)."""
    if lam == 0:
        return rho
    k = len(qubits)
    t = rho.reshape([2] * (2 * n))
    row_axes = [n - 1 - q for q in qubits]
    col_axes = [2 * n - 1 - q for q in qubits]
    letters = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    sub = list(letters[:2 * n])
    for ra, ca in zip(row_axes, col_axes):
        sub[ca] = sub[ra]
    keep = [i for i in range(2 * n) if i not in row_axes + col_axes]
    red = np.einsum("".join(sub) + "->" + "".join(sub[i] for i in keep), t)
    eye = np.eye(2 ** k).reshape([2] * (2 * k))      # identity: row bit i pairs with column bit i
    full = np.multiply.outer(red, eye)
    full = np.moveaxis(full, list(range(len(keep), 2 * n)), row_axes[::-1] + col_axes[::-1])
    return (1 - lam) * rho + lam * full.reshape(2 ** n, 2 ** n) / 2 ** k


def exact_probabilities(ops, n, measured, p2, readout):
    """Exact probabilities of the recorded results under the course noise model.

    ops: list of (name, qubits, params) applied to |0...0>; measured: the qubits measured at the end, measured[j] into
    classical bit j. Returns an array over the 2^len(measured) recorded results (bit j of the index is classical bit j).
    """
    rho = np.zeros((2 ** n, 2 ** n), dtype=complex)
    rho[0, 0] = 1
    for name, qubits, params in ops:
        rho = _apply(rho, _matrix(name, params), list(qubits), n)
        if name in TWO_QUBIT_GATES:
            rho = _depolarize(rho, p2, list(qubits), n)
        elif name in ONE_QUBIT_GATES:
            rho = _depolarize(rho, p2 / 10, list(qubits), n)
    probs = np.real(np.diag(rho)).clip(min=0)
    probs = probs / probs.sum()
    m = len(measured)
    idx = np.arange(2 ** n)
    code = np.zeros(2 ** n, dtype=int)
    for j, q in enumerate(measured):
        code |= ((idx >> q) & 1) << j
    dist = np.zeros(2 ** m)
    np.add.at(dist, code, probs)
    conf = np.array([[1 - readout, readout], [2 * readout, 1 - 2 * readout]])
    t = dist.reshape([2] * m)
    for j in range(m):
        ax = m - 1 - j
        t = np.moveaxis(np.tensordot(t, conf, axes=([ax], [0])), -1, ax)
    dist = t.reshape(-1).clip(min=0)
    return dist / dist.sum()


def build_circuit(ops, n, measured):
    """The ops as a QuantumCircuit (Qiskit or qsim), with the measurements at the end."""
    qc = QuantumCircuit(n, len(measured)) if measured else QuantumCircuit(n)
    for name, qubits, params in ops:
        getattr(qc, name)(*params, *qubits)
    for j, q in enumerate(measured):
        qc.measure(q, j)
    return qc


# ---------------------------------------------------------------- helpers for the checks
def _op(inst):
    return getattr(inst, "operation", inst)


def _name(inst):
    return getattr(_op(inst), "name", "")


def _qubit_indices(qc, inst):
    qs = getattr(inst, "qubits", ())
    try:
        return [qc.find_bit(q).index for q in qs]
    except Exception:  # noqa: BLE001
        return [int(q) for q in qs]


def _clbit_indices(qc, inst):
    cs = getattr(inst, "clbits", ())
    try:
        return [qc.find_bit(c).index for c in cs]
    except Exception:  # noqa: BLE001
        return [int(c) for c in cs]


def measurement_map(qc):
    """{qubit: classical bit} for the measurements in a circuit."""
    return {_qubit_indices(qc, i)[0]: _clbit_indices(qc, i)[0] for i in qc.data if _name(i) == "measure"}


def big_gates(qc):
    """Names of gates on 3 or more qubits (the course noise model puts no error on them)."""
    return sorted({_name(i) for i in qc.data if _name(i) not in ("barrier", "measure") and len(_qubit_indices(qc, i)) >= 3})


def two_qubit_count(qc):
    """The number of two-qubit gates in a circuit (barriers not counted)."""
    return sum(1 for i in qc.data if _name(i) not in ("barrier", "measure") and len(_qubit_indices(qc, i)) == 2)


def ops_two_qubit_count(ops):
    return sum(1 for name, qubits, params in ops if len(qubits) == 2)


def ideal_probabilities(qc):
    """Probabilities of all qubits' values (index bit q = qubit q) for a circuit without its final measurements."""
    c = qc.copy()
    c.remove_final_measurements()
    return np.abs(np.asarray(Statevector(c).data, dtype=complex)) ** 2


def counts_total(counts):
    try:
        return int(sum(int(v) for v in counts.values()))
    except Exception:  # noqa: BLE001
        return -1


# ---------------------------------------------------------------- the statistics functions
def _close(a, b, tol=1e-9):
    try:
        return abs(float(a) - float(b)) <= tol * max(1.0, abs(float(b)))
    except (TypeError, ValueError):
        return False


def test_statistics(shot_sigma, within_3_sigma, difference_sigma):
    """Returns (passed, messages)."""
    msgs = []
    cases = [(0.5, 100), (0.687, 4000), (0.1, 1000), (0.0, 50), (1.0, 50), (0.9206, 4000)]
    for p, n in cases:
        want = math.sqrt(p * (1 - p) / n)
        try:
            got = shot_sigma(p, n)
        except NotImplementedError:
            return False, ["shot_sigma() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"shot_sigma({p}, {n}) raised {type(e).__name__}: {e}"]
        if not _close(got, want):
            hint = ""
            if _close(got, math.sqrt(p * (1 - p) * n)):
                hint = " That is the uncertainty of the count k; divide by shots for the fraction: sqrt(p * (1 - p) / shots)."
            elif _close(got, p * (1 - p) / n):
                hint = " Take the square root: sqrt(p * (1 - p) / shots)."
            elif _close(got, math.sqrt(p / n)) or _close(got, math.sqrt(1 / n)):
                hint = " Use p * (1 - p), the binomial variance of one shot."
            return False, [f"shot_sigma({p}, {n}) returned {got}; expected {want:.6g}.{hint}"]
    msgs.append("shot_sigma(): passed (6 cases).")

    w_cases = [(2748, 4000, 0.687, True), (2655, 4000, 0.687, False), (2841, 4000, 0.687, False),
               (2700, 4000, 0.687, True), (2821, 4000, 0.687, True), (55, 100, 0.5, True), (66, 100, 0.5, False),
               (34, 100, 0.5, False), (3700, 4000, 0.9206, True), (3620, 4000, 0.9206, False)]
    for k, n, p, want in w_cases:
        try:
            got = within_3_sigma(k, n, p)
        except NotImplementedError:
            return False, msgs + ["within_3_sigma() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, msgs + [f"within_3_sigma({k}, {n}, {p}) raised {type(e).__name__}: {e}"]
        if bool(got) != want or not isinstance(got, (bool, np.bool_)):
            d = abs(k / n - p) / max(math.sqrt(p * (1 - p) / n), 1e-300)
            hint = ""
            if not isinstance(got, (bool, np.bool_)):
                hint = " Return True or False."
            elif k > n * p and not want:
                hint = " Check both directions: a result can be too high as well as too low (use abs())."
            elif 2 < d <= 3:
                hint = " The course rule allows up to 3 standard deviations."
            else:
                hint = " Compare the fraction k / shots with p, using shot_sigma(p, shots)."
            return False, msgs + [f"within_3_sigma({k}, {n}, {p}) returned {got}; expected {want} "
                                  f"(the result is {d:.2f} standard deviations from p).{hint}"]
    msgs.append("within_3_sigma(): passed (10 cases).")

    d_cases = [(0.687, 0.634, 4000), (0.5, 0.5, 100), (0.9, 0.1, 1000), (0.2, 0.25, 4000)]
    for f1, f2, n in d_cases:
        want = math.sqrt(f1 * (1 - f1) / n + f2 * (1 - f2) / n)
        try:
            got = difference_sigma(f1, f2, n)
        except NotImplementedError:
            return False, msgs + ["difference_sigma() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, msgs + [f"difference_sigma({f1}, {f2}, {n}) raised {type(e).__name__}: {e}"]
        if not _close(got, want):
            s1, s2 = math.sqrt(f1 * (1 - f1) / n), math.sqrt(f2 * (1 - f2) / n)
            hint = ""
            if _close(got, s1 + s2):
                hint = " Uncertainties add in quadrature: sqrt(sigma1**2 + sigma2**2), not sigma1 + sigma2."
            elif _close(got, abs(s1 - s2)):
                hint = " The uncertainty of a difference is larger than either one: sqrt(sigma1**2 + sigma2**2)."
            elif _close(got, want ** 2):
                hint = " Take the square root."
            return False, msgs + [f"difference_sigma({f1}, {f2}, {n}) returned {got}; expected {want:.6g}.{hint}"]
    msgs.append("difference_sigma(): passed (4 cases).")
    return True, msgs


def personal_noise_ok(P2, READOUT):
    """Checks the personal noise numbers from Canvas; returns (ok, message, p2, readout)."""
    try:
        p2, ro = float(P2), float(READOUT)
    except (TypeError, ValueError):
        return False, "P2 and READOUT must be the numbers shown in your Canvas project quiz.", None, None
    if not (0.005 - 1e-12 <= p2 <= 0.030 + 1e-12) or abs(round(p2, 3) - p2) > 1e-9:
        return False, "P2 is a number from 0.005 to 0.030 with three decimals. Copy it again from the project quiz.", None, None
    if not (0.005 - 1e-12 <= ro <= 0.030 + 1e-12) or abs(round(ro, 3) - ro) > 1e-9:
        return False, "READOUT is a number from 0.005 to 0.030 with three decimals. Copy it again from the project quiz.", None, None
    return True, "", round(p2, 3), round(ro, 3)


PROJECT_EDGES = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0), (0, 2)]
PROJECT_N = 5
REFERENCE_ANGLES = {1: ([2.817], [0.354]),
                    2: ([2.849, 2.627], [0.517, 0.309]),
                    3: ([2.905, 2.669, 2.619], [0.535, 0.401, 0.204])}


def cut_of(index, edges):
    """Cut size of the split whose node q is bit q of the integer index."""
    return sum(((index >> i) & 1) != ((index >> j) & 1) for i, j in edges)


def reference_qaoa_ops(gammas, betas, edges, n):
    ops = [("h", [q], []) for q in range(n)]
    for g, b in zip(gammas, betas):
        ops += [("rzz", [i, j], [2 * g]) for i, j in edges]
        ops += [("rx", [q], [2 * b]) for q in range(n)]
    return ops


def reference_qaoa(gammas, betas, edges, n):
    return build_circuit(reference_qaoa_ops(gammas, betas, edges, n), n, [])


def noisy_distribution(layers, p2, readout):
    g, b = REFERENCE_ANGLES[layers]
    return exact_probabilities(reference_qaoa_ops(g, b, PROJECT_EDGES, PROJECT_N), PROJECT_N, list(range(PROJECT_N)), p2, readout)


def _cuts():
    return np.array([cut_of(k, PROJECT_EDGES) for k in range(2 ** PROJECT_N)])


def expected_cut(layers, p2, readout):
    return float(noisy_distribution(layers, p2, readout) @ _cuts())


def cut_sigma(layers, p2, readout, shots=SHOTS):
    """The standard deviation of the mean cut of `shots` shots (the spread of one shot's cut, divided by sqrt(shots))."""
    d = noisy_distribution(layers, p2, readout)
    c = _cuts()
    mean = d @ c
    return float(math.sqrt(d @ (c - mean) ** 2 / shots))


def expected_max_cut_probability(layers, p2, readout):
    d = noisy_distribution(layers, p2, readout)
    c = _cuts()
    return float(d[c == c.max()].sum())


QAOA_CASES = [([0.4], [0.7], [(0, 1)], 2), ([2.817], [0.354], PROJECT_EDGES, 5),
              ([0.3, 0.9], [1.1, 0.2], [(0, 1), (1, 2), (0, 2)], 3), ([2.849, 2.627], [0.517, 0.309], PROJECT_EDGES, 5)]


def _test_qaoa(qaoa_circuit):
    for gammas, betas, edges, n in QAOA_CASES:
        label = f"qaoa_circuit({gammas}, {betas}, edges, {n})"
        try:
            qc = qaoa_circuit(list(gammas), list(betas), list(edges), n)
        except NotImplementedError:
            return False, ["qaoa_circuit() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"{label} raised {type(e).__name__}: {e}"]
        if not hasattr(qc, "data") or getattr(qc, "num_qubits", None) != n:
            return False, [f"{label} should return a QuantumCircuit on {n} qubits."]
        if measurement_map(qc):
            return False, [f"Leave measurements out of {label}, as in Module 5: the training uses the state itself, and "
                           "the prepared measured(qc) adds the measurements for the runs with shots."]
        try:
            got = ideal_probabilities(qc)
        except Exception as e:  # noqa: BLE001
            return False, [f"The circuit from {label} cannot be simulated: {type(e).__name__}: {e}"]
        want = ideal_probabilities(reference_qaoa(gammas, betas, edges, n))
        if np.abs(got - want).max() > 1e-9:
            half = ideal_probabilities(reference_qaoa([g / 2 for g in gammas], [b / 2 for b in betas], edges, n))
            hint = " Check each part: H on every qubit; then, for each layer, rzz(2 * gamma) on every edge and rx(2 * beta) on every qubit."
            if np.abs(got - half).max() < 1e-9:
                hint = " Use twice the angles: rzz(2 * gamma, i, j) and rx(2 * beta, q)."
            return False, [f"{label} gives probabilities different from the expected ones.{hint}"]
        n2, want2 = two_qubit_count(qc), len(edges) * len(gammas)
        if n2 != want2:
            return False, [f"{label} gives the right probabilities but has {n2} two-qubit gates; the project's circuit has "
                           f"{want2} (one rzz per edge in each layer). With noise, every extra gate changes the result."]
    return True, [f"qaoa_circuit(): passed ({len(QAOA_CASES)} cases, 1 and 2 layers, 2 to 5 nodes)."]


CUT_CASES = [({"01001": 10}, PROJECT_EDGES), ({"00000": 1, "11111": 1}, PROJECT_EDGES),
             ({"10": 3, "01": 1, "00": 4}, [(0, 1)]), ({"001": 2, "011": 1, "111": 1}, [(0, 1), (1, 2), (0, 2)]),
             ({"10110": 7, "00001": 2, "11000": 1}, PROJECT_EDGES)]


def _test_mean_cut(mean_cut):
    for counts, edges in CUT_CASES:
        want = sum(v * cut_of(int(k, 2), edges) for k, v in counts.items()) / sum(counts.values())
        try:
            got = mean_cut(dict(counts), list(edges))
        except NotImplementedError:
            return False, ["mean_cut() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"mean_cut({counts}, {edges}) raised {type(e).__name__}: {e}"]
        try:
            gotf = float(got)
        except (TypeError, ValueError):
            return False, [f"mean_cut({counts}, edges) should return a number; it returned {got!r}."]
        if abs(gotf - want) > 1e-9:
            rev = sum(v * cut_of(int(k[::-1], 2), edges) for k, v in counts.items()) / sum(counts.values())
            unw = np.mean([cut_of(int(k, 2), edges) for k in counts])
            tot = sum(v * cut_of(int(k, 2), edges) for k, v in counts.items())
            hint = ""
            if abs(gotf - rev) < 1e-9:
                hint = " Node q is the bit q places from the right: key[-1 - q], or bit q of int(key, 2)."
            elif abs(gotf - unw) < 1e-9:
                hint = " Weight each result by its count."
            elif abs(gotf - tot) < 1e-9:
                hint = " Divide by the total number of shots."
            return False, [f"mean_cut({counts}, {edges}) returned {gotf:.6f}; expected {want:.6f}.{hint}"]
    return True, [f"mean_cut(): passed ({len(CUT_CASES)} cases)."]


def _test_counts(my_counts, layers, p2, ro):
    if not isinstance(my_counts, dict) or not my_counts:
        return False, ["my_counts is empty. Run the cell of Step 8 that makes it first."]
    total = counts_total(my_counts)
    if total != SHOTS:
        return False, [f"my_counts holds {total} shots; use shots=4000 (Step 8)."]
    if any(len(str(k).replace(" ", "")) != PROJECT_N for k in my_counts):
        return False, ["The keys of my_counts should be 5-bit strings, one bit per node of the project graph."]
    c = sum(int(v) * cut_of(int(str(k).replace(" ", ""), 2), PROJECT_EDGES) for k, v in my_counts.items()) / SHOTS
    e, s = expected_cut(layers, p2, ro), cut_sigma(layers, p2, ro)
    z = (c - e) / s
    if abs(z) > 3:
        return False, [f"my_counts give a mean cut of {c:.4f}, {z:+.1f} standard deviations from the noise model's expected "
                       f"cut {e:.4f} for {layers} layer(s). Check that my_counts comes from qaoa_circuit with the reference "
                       "angles for LAYERS, run with course_noise_model(P2, READOUT) and 4,000 shots. If it does, a result this "
                       "far out happens by chance about 3 times in 1,000: run Step 8 again with another seed."]
    return True, [f"my_counts: mean cut {c:.4f}, within 3 standard deviations ({z:+.2f}) of the expected {e:.4f}."]


def personal_value(layers, p2, readout):
    return int(round(1000 * expected_cut(int(layers), float(p2), float(readout))))


def check_l2_m6_qaoa(qaoa_circuit, mean_cut, shot_sigma, within_3_sigma, difference_sigma, LAYERS, P2, READOUT, my_counts):
    try:
        layers = int(LAYERS)
        if layers != float(LAYERS) or layers not in REFERENCE_ANGLES:
            raise ValueError
    except (TypeError, ValueError):
        return False, ["LAYERS is 1, 2 or 3. Copy it again from your project quiz."], None
    ok, msg, p2, ro = personal_noise_ok(P2, READOUT)
    if not ok:
        return False, [msg], None
    msgs, passed = [], True
    for good, mm in (test_statistics(shot_sigma, within_3_sigma, difference_sigma), _test_qaoa(qaoa_circuit),
                     _test_mean_cut(mean_cut)):
        msgs += mm
        passed = passed and good
        if not good:
            break
    if not passed:
        return False, msgs, None
    good, mm = _test_counts(my_counts, layers, p2, ro)
    msgs += mm
    if not good:
        return False, msgs, None
    return True, msgs, int(round(1000 * expected_cut(layers, p2, ro)))


passed, messages, value = check_l2_m6_qaoa(qaoa_circuit, mean_cut, shot_sigma, within_3_sigma, difference_sigma,
                                           LAYERS, P2, READOUT, my_counts)
for m in messages:
    print(m)
if passed:
    print("\nAll tests passed. Your verification value is", value)
else:
    print("\nNot yet: fix the item above and run this cell again.")

## Limits of this experiment

- **Gates as a real computer runs them.** Here `rzz` is one two-qubit gate with one error. IBM computers run it as `cz` gates and single-qubit rotations, and the triangle (0, 1, 2) cannot be placed on a heavy-hex chip without swaps (Module 4): each layer would cost more two-qubit gates, and the crossover would come at a lower P2.
- **Training without noise.** For depolarizing noise, the noise mostly shrinks the landscape toward the random-split value without moving its peak, so angles trained without noise stay near the best. Errors that rotate the state in the same direction every time do move the peak; training on the computer itself, or with a more detailed noise model, can then find better angles.
- **A model, not a computer.** The same error on every gate and qubit, no drift, no crosstalk between neighbouring qubits.
- **One small graph.** With 5 nodes, brute force is instant. QAOA is only interesting for graphs far too large for brute force, where no one knows the maximum cut to compare with.

These limits are not failures of your experiment: every experiment has them. Saying what a result does and does not show is part of reporting it.

## Optional: run it on a real IBM quantum computer

If you have an IBM Quantum account with an Open Plan instance (see **Set Up Your Tools** in Start Here), the [Module 6 hardware notebook in Colab](https://colab.research.google.com/github/CSU-Physics/quantum-computing-notebooks/blob/main/colab/level2/QC-L2-M6-hardware-colab.ipynb) runs the project's reference circuit on a real computer and compares the result with the ideal value, with this course's noise model, and with IBM's own noise model of that computer. Your API key stays in Colab's **Secrets**; never type it into a cell. The Open Plan gives up to 10 minutes of quantum computer time every 28 days, and one run of 4,000 shots uses a few seconds of it, but the queue can take from seconds to hours. This step is not graded, and the badge does not depend on it.

## Optional: a short summary of your experiment

Not graded. Writing five short answers is a good way to prepare for the project quiz, and it is how results are reported in a lab notebook or a paper. Double-click this cell to edit it.

1. **Question.** What did you want to find out?
2. **Method.** Which circuit, which noise model (P2, READOUT), how many shots?
3. **Result.** Your main numbers, each with its uncertainty (for example 0.687 ± 0.007, with σ from Step 1).
4. **Interpretation.** What do the numbers say about the question? Which differences are significant (more than 3σ)?
5. **Limits.** What does this experiment not show?

**Next in Canvas:** the Module 6 project quiz for this experiment, QAOA with noise, and the Module 6 time log.